In [10]:
# =========================
# 1. IMPORTS
# =========================
import torch
from tqdm import tqdm
import numpy as np

# If using torchvision detection models
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn

# =========================
# 2. DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [11]:
import torch.nn as nn

class CustomROIHead(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()

        self.fc1 = nn.Linear(in_features, 1024)
        self.bn1 = nn.BatchNorm1d(1024)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(0.3)

        self.fc2 = nn.Linear(1024, 1024)
        self.bn2 = nn.BatchNorm1d(1024)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(0.3)

        self.cls_score = nn.Linear(1024, num_classes)
        self.bbox_pred = nn.Linear(1024, num_classes * 4)

    def forward(self, x):
        x = x.flatten(start_dim=1)

        x = self.drop1(self.relu1(self.bn1(self.fc1(x))))
        x = self.drop2(self.relu2(self.bn2(self.fc2(x))))

        return self.cls_score(x), self.bbox_pred(x)

In [12]:
import os
import json
import torch
import cv2
from torch.utils.data import Dataset

CLASS_MAP = {
    "speedlimit": 1,
    "crosswalk": 2,
    "trafficlight": 3,
    "stop": 4
}

IDX_TO_CLASS = {v: k for k, v in CLASS_MAP.items()}

class NinjaDataset(Dataset):
    def __init__(self, img_dir, ann_dir):
        self.img_dir = img_dir
        self.ann_dir = ann_dir

        self.images = sorted([
            f for f in os.listdir(img_dir)
            if f.endswith(".png")
        ])

    def __getitem__(self, idx):
        img_name = self.images[idx]

        img_path = os.path.join(self.img_dir, img_name)
        ann_path = os.path.join(self.ann_dir, img_name + ".json")

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        boxes = []
        labels = []

        if os.path.exists(ann_path):
            with open(ann_path) as f:
                data = json.load(f)

            for obj in data.get("objects", []):
                x1, y1 = obj["points"]["exterior"][0]
                x2, y2 = obj["points"]["exterior"][1]

                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)

                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(CLASS_MAP[obj["classTitle"]])

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx])
        }

        img = torch.tensor(img / 255.0, dtype=torch.float32).permute(2, 0, 1)

        return img, target

    def __len__(self):
        return len(self.images)

In [13]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    return tuple(zip(*batch))

dataset = NinjaDataset(
    img_dir="D:/617 project/raw_data/road-sign-detection-DatasetNinja/ds/img",
    ann_dir="D:/617 project/raw_data/road-sign-detection-DatasetNinja/ds/ann"
)

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn
)

In [14]:
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = 5  # MUST match training

model = fasterrcnn_resnet50_fpn(weights=None)

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = CustomROIHead(in_features, num_classes)

model.load_state_dict(
    torch.load("D:/617 project/faster_R_CNN/best_model_final.pth", map_location=device)
)

model.to(device)
model.eval();

In [15]:
# import json
# from tqdm import tqdm

# output_file = "predictions5.jsonl"
# score_threshold = 0.7  # adjust if needed

# with open(output_file, "w") as f:
#     with torch.no_grad():
#         for idx, (images, targets) in enumerate(tqdm(loader)):
#             images = [img.to(device) for img in images]

#             outputs = model(images)

#             for i, output in enumerate(outputs):
#                 boxes = output["boxes"].cpu().numpy()
#                 scores = output["scores"].cpu().numpy()
#                 labels = output["labels"].cpu().numpy()

#                 # ✅ filter low-confidence boxes
#                 keep = scores > score_threshold

#                 boxes = boxes[keep]
#                 scores = scores[keep]
#                 labels = labels[keep]

#                 record = {
#                     "image_id": idx * len(outputs) + i,
#                     "boxes": boxes.tolist(),
#                     "scores": scores.tolist(),
#                     "labels": labels.tolist()
#                 }

#                 f.write(json.dumps(record) + "\n")

In [16]:
import json

preds = []
with open("frcnn_test_on_ninja_dataset_result.jsonl") as f:
    for line in f:
        preds.append(json.loads(line))

print("Loaded:", len(preds))

Loaded: 877


In [17]:
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)

    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

    union = area1 + area2 - inter

    return inter / union if union > 0 else 0

In [18]:
iou_threshold = 0.75

TP = 0  # true positives
FP = 0  # false positives
FN = 0  # false negatives

for i in range(len(dataset)):
    img, target = dataset[i]

    gt_boxes = target["boxes"].numpy()
    gt_labels = target["labels"].numpy()

    pred = preds[i]
    pred_boxes = pred["boxes"]
    pred_labels = pred["labels"]

    matched_gt = set()

    # match predictions to GT
    for p_box, p_label in zip(pred_boxes, pred_labels):
        best_iou = 0
        best_gt_idx = -1

        for j, (g_box, g_label) in enumerate(zip(gt_boxes, gt_labels)):
            if j in matched_gt:
                continue

            if p_label != g_label:
                continue

            iou = compute_iou(p_box, g_box)

            if iou > best_iou:
                best_iou = iou
                best_gt_idx = j

        if best_iou >= iou_threshold:
            TP += 1
            matched_gt.add(best_gt_idx)
        else:
            FP += 1

    FN += len(gt_boxes) - len(matched_gt)

In [19]:
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0
    else 0
)

accuracy = TP / (TP + FP + FN) if (TP + FP + FN) > 0 else 0

print(f"TP: {TP}, FP: {FP}, FN: {FN}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Accuracy:  {accuracy:.4f}")

TP: 1232, FP: 26, FN: 12
Precision: 0.9793
Recall:    0.9904
F1 Score:  0.9848
Accuracy:  0.9701
